# Notebook Task — Credit Installments Ingestion

**Objective:** Read synthetic credit installment records from CSV files in a Unity Catalog volume and persist them in a staging Delta table using PySpark.

## Why?

**For our project:** Creates `installments_staging` as the third ingestion output, using `catalog` and `schema` task parameters. Lakeflow Jobs will execute this notebook within a job configured with a file arrival trigger.

**For Data Engineering:** Demonstrates parameterized ingestion, governed file access through Unity Catalog, and separation between data processing and workflow orchestration.

#### 01 — Configure Parameters and Define the Volume Path

Read the catalog and schema parameters and build the volume path used by the file arrival trigger. Defaults allow manual validation before configuring the task parameters.

In [ ]:
# Define defaults for manual notebook execution.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "fintech_jobs")

# Read the current parameter values.
my_catalog = dbutils.widgets.get("catalog")
my_schema = dbutils.widgets.get("schema")

# Build the Unity Catalog volume path.
my_volume_path = (
    f"/Volumes/{my_catalog}/{my_schema}/trigger_storage_location/"
)

# Display the configuration for validation.
print(my_catalog)
print(my_schema)
print(my_volume_path)

workspace
fintech_jobs
/Volumes/workspace/fintech_jobs/trigger_storage_location/


#### 02 — Ingest Credit Installments into Staging

Read installment CSV files from the configured volume and overwrite the staging Delta table with their contents.

In [ ]:
# Read installment CSV files, using the first row as column headers.
installments_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(my_volume_path)
)

# Define the staging table using the notebook parameters.
target_table = f"{my_catalog}.{my_schema}.installments_staging"

# Persist the records as a Delta table.
(
    installments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

#### 03 — Preview Credit Installments

Display the records loaded from the CSV files to inspect the ingestion result.

In [ ]:
# Display the installment records loaded during ingestion.
display(installments_df)

installment_id,application_id,customer_id,due_date,amount_due,payment_status
3001,2001,101,2026-10-20,1750.00,PAID
3002,2001,101,2026-11-20,1750.00,PENDING
3003,2004,104,2026-10-25,4000.00,PAID
3004,2004,104,2026-11-25,4000.00,PENDING
3005,2004,104,2026-12-25,4000.00,PENDING
3006,2005,105,2026-11-10,2600.00,PENDING
